<a href="https://colab.research.google.com/github/somyamaheshwari2612/flyyyyy/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/somyamaheshwari2612/flyyyyy/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window
Unit of Analysis: One row represents one pseudonymized content item (a specific web page) for a single day.

Time Window: I am restricting this analysis to a mid-panel month, specifically March 2026 (month='2026-03'), to ensure I am not accidentally testing on the final month of data.



In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded
Label (Proxy): is_declining (derived from trend_direction == 'down'). This is what the model attempts to predict.

Features: impressions, clicks, ga4_sessions, ga4_total_engagement_sec, and sessions_ai. These are all knowable at the decision moment because they are historical, observable measurements recorded before any editorial review takes place.

Context: content_hash_id and client_hash_id. These are used purely for grouping and joining tables, not as predictive signals.

Excluded: health_score, priority_score, and action_type. I am deliberately excluding these because they are FlyRank's pre-calculated product decision flags. Using them would cause circular logic (target leakage), where the model simply memorizes the existing rule.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

Five Features (Knowable at the decision moment):

impressions: Knowable because it is a historical count of search visibility prior to the review date.

clicks: Knowable because it is a historical measurement of user engagement prior to the review date.

ga4_sessions: Knowable because it is a historical count of site visits prior to the review date.

ga4_total_engagement_sec: Knowable because it historically tracks how long users interacted with the page.

sessions_ai: Knowable because it is a historical measurement of traffic referred from AI platforms.

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [9]:

import duckdb
from google.colab import userdata
import pandas as pd

# 1. Initialize DuckDB and authenticate using the Colab Secret you set up
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{userdata.get('HF_TOKEN')}');")

# Define the Hugging Face path for the March 2026 data partition
hf_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# --- QUERY 1: Check Row Count and Date Span ---
print("--- Row Count & Date Span ---")
query_span = f"""
    SELECT
        COUNT(*) as total_rows,
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM read_parquet('{hf_path}')
"""
display(con.execute(query_span).df())

# --- QUERY 2: Verify the Grain (One row = one content item per day) ---
print("\n--- Grain Verification ---")
query_grain = f"""
    SELECT
        report_date,
        content_hash_id,
        COUNT(*) as row_count
    FROM read_parquet('{hf_path}')
    GROUP BY report_date, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
"""
# If the dataframe is empty, it proves there are no duplicates and our grain is correct!
display(con.execute(query_grain).df())

# --- QUERY 3: Availability Check (using IS TRUE) ---
print("\n--- GA4 Data Availability ---")
query_avail = f"""
    SELECT COUNT(*) as valid_ga4_rows
    FROM read_parquet('{hf_path}')
    WHERE ga4_data_available IS TRUE
"""
display(con.execute(query_avail).df())
# --- THE 5 FEATURES FRAME ---
print("\n--- 5 Features Frame ---")
query_features = f"""
    SELECT
        content_hash_id,
        gsc_impressions AS impressions,
        gsc_clicks AS clicks,
        ga4_sessions,
        ga4_total_engagement_sec,
        sessions_ai
    FROM read_parquet('{hf_path}')
    WHERE ga4_data_available IS TRUE
    LIMIT 5
"""
df_features = con.execute(query_features).df()
display(df_features)

# --- THE TRAP (Leakage Experiment) ---
print("\n--- The Trap (Leakage Column) ---")
# 1. Add the trap (a label-derived column)
df_features['TRAP_leaky_target_copy'] = (df_features['impressions'] < 100).astype(int)
print("Trap Added: 'TRAP_leaky_target_copy' (Simulates 100% accuracy jump)")
display(df_features.head(2))

# 2. Delete the trap to keep the model honest
df_features = df_features.drop(columns=['TRAP_leaky_target_copy'])
print("Trap Removed: Dropped the leaky column to prevent circular logic.")

--- Row Count & Date Span ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31



--- Grain Verification ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,impressions,clicks,ga4_sessions,ga4_total_engagement_sec,sessions_ai
0,content_09be8cc7fcb222af,0,0,1,0,0
1,content_851afac9fe13612e,0,0,1,0,0
2,content_cee6c6fc8c51af14,0,0,1,0,0
3,content_5e120e972f11f833,0,0,1,0,0
4,content_16a7291bb6ecaebe,0,0,1,0,0



--- The Trap (Leakage Column) ---
Trap Added: 'TRAP_leaky_target_copy' (Simulates 100% accuracy jump)


,content_hash_id,impressions,clicks,ga4_sessions,ga4_total_engagement_sec,sessions_ai,TRAP_leaky_target_copy
0,content_09be8cc7fcb222af,0,0,1,0,0,1
1,content_851afac9fe13612e,0,0,1,0,0,1


Trap Removed: Dropped the leaky column to prevent circular logic.


## 4. Data limits
Limitation 1: Unbalanced Panel. The history is not uniform across all clients. Because ga4_data_start varies by client, earlier rows contain only search data. Time-windows must carefully account for this so we don't confuse "no tracking installed yet" with "zero traffic."

Limitation 2: No Semantic Text. Because the raw text, titles, and URLs have been heavily pseudonymized for privacy, I cannot perform true semantic clustering or NLP analysis on the actual content of the pages.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [Yes ] Every section above is filled — markdown thinking AND the code that backs it
- [Yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ Yes] No client names, URLs, or private queries anywhere
- [Yes ] My claims use careful words: observed, measured, directional, decision-support
- [ Yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.